In [3]:
from statsmodels.tsa.stattools import adfuller
import pandas as pd
import numpy as np
from special_group import special_group

filename = 'Var/new_table_2017-03-15.csv'
data = pd.read_csv(f'{filename}')

result = special_group(data, group_amount=500)
result.head()

d:\OneDrive\Documents\Stud\S6\ST4  Modèle et agent en finance\st4-ei2-groupe2\special_group.py:41: UserWarning: Unexpected column found: bid_1
  warnings.warn(f"Unexpected column found: {col}")
d:\OneDrive\Documents\Stud\S6\ST4  Modèle et agent en finance\st4-ei2-groupe2\special_group.py:41: UserWarning: Unexpected column found: ask_1
  warnings.warn(f"Unexpected column found: {col}")


,datetime,V_lo_b,V_c_b,V_ex_b,V_lo_a,V_c_a,V_ex_a,mid_price,returns,deltaT
0,2017-03-15 09:32:36,24955.0,29647.0,0.0,33121.0,28525.0,537.0,46.571600,-0.000012,33.0
1,2017-03-15 09:33:25,30005.0,35084.0,1735.0,28485.0,21888.0,1365.0,46.573385,0.000038,49.0
2,2017-03-15 09:34:12,35681.0,30042.0,191.0,25102.0,26477.0,2370.0,46.562430,-0.000235,47.0
3,2017-03-15 09:34:43,43400.0,24883.0,0.0,18226.0,31414.0,850.0,46.591920,0.000633,31.0
4,2017-03-15 09:35:00,41806.0,25085.0,0.0,19173.0,29126.0,1748.0,46.632575,0.000872,17.0


In [4]:
# Sélection des colonnes numériques (hors "datetime" ou colonnes non numériques)
cols_to_test = result.select_dtypes(include='number').columns

results_summary = []
# Fonction de test ADF
def test_stationarity(series, name, label="original"):
    series_clean = pd.Series(series).dropna() 
    adf_res = adfuller(series_clean)
    adf_stat = adf_res[0]
    p_value = adf_res[1]
    stationnaire = p_value < 0.05
    results_summary.append({
        "colonne": name,
        "version": label,
        "ADF stat": adf_stat,
        "p-value": p_value,
        "stationnaire": stationnaire
    })
    return stationnaire

In [5]:
# Boucle sur les colonnes
for col in cols_to_test:
    print(f"Test ADF pour la colonne '{col}' :")
    series = result[col]
    is_stat = test_stationarity(series, col, label="original")

    if not is_stat:
        print(f"   Non stationnaire → test après np.diff()")
        diff_series = np.diff(series)
        is_stat_diff = test_stationarity(diff_series, col, label="diff")
        if is_stat_diff:
            print(f"   Stationnaire après différenciation")
        else:
            print(f"   Toujours non stationnaire après différenciation")
    else:
        print(f"   Déjà stationnaire")

# Affichage du résumé
results_df = pd.DataFrame(results_summary)
print("\n Résumé des résultats ADF :")
print(results_df)

Test ADF pour la colonne 'V_lo_b' :
   Déjà stationnaire
Test ADF pour la colonne 'V_c_b' :
   Déjà stationnaire
Test ADF pour la colonne 'V_ex_b' :
   Déjà stationnaire
Test ADF pour la colonne 'V_lo_a' :
   Déjà stationnaire
Test ADF pour la colonne 'V_c_a' :
   Déjà stationnaire
Test ADF pour la colonne 'V_ex_a' :
   Déjà stationnaire
Test ADF pour la colonne 'mid_price' :
   Non stationnaire → test après np.diff()
   Stationnaire après différenciation
Test ADF pour la colonne 'returns' :
   Déjà stationnaire
Test ADF pour la colonne 'deltaT' :
   Non stationnaire → test après np.diff()
   Stationnaire après différenciation

 Résumé des résultats ADF :
      colonne   version   ADF stat       p-value  stationnaire
0      V_lo_b  original -17.996171  2.743942e-30          True
1       V_c_b  original  -7.614439  2.210940e-11          True
2      V_ex_b  original  -3.410355  1.061088e-02          True
3      V_lo_a  original  -4.631301  1.130555e-04          True
4       V_c_a  origin